### Data Ingestion

In [1]:
### Document Structure

from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="This is the main content of the document. I am learning how to use RAG.",
    metadata={"source": "example.txt", "page": 1, "author": "Mohd Ahkam", "Date": "2024-06-01"}
)
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'Mohd Ahkam', 'Date': '2024-06-01'}, page_content='This is the main content of the document. I am learning how to use RAG.')

In [3]:
## Creating a simple text file
import os 
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_text = {
    "../data/text_files/python_intro.txt":"""This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework."""}
for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
print("sample text file created successfully.")

sample text file created successfully.


In [5]:
### Text Loader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]


In [6]:
## Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## Load all text files in the directory
directory_loader = DirectoryLoader(
    "../data/text_files", glob="**/*.txt", ##pattern to match files
    loader_cls=TextLoader, ##Loader class to use
    loader_kwargs={'encoding':"utf-8"},
    show_progress=False
    
)
documents = directory_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]

In [7]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader

## Load all pdf files in the directory
directory_loader = DirectoryLoader(
    "../data/pdf", glob="**/*.pdf", ##pattern to match files
    loader_cls=PyMuPDFLoader, ##Loader class to use
    show_progress=False
    
)
pdf_documents = directory_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-08-12T20:56:23+05:30', 'source': '..\\data\\pdf\\Ahkam web Exp.pdf', 'file_path': '..\\data\\pdf\\Ahkam web Exp.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'aneesur rahman', 'subject': '', 'keywords': '', 'moddate': '2024-08-12T20:56:23+05:30', 'trapped': '', 'modDate': "D:20240812205623+05'30'", 'creationDate': "D:20240812205623+05'30'", 'page': 0}, page_content='TO WHOM SO EVER IT MAY CONCERN \n \n \nREF: VOL/EL/R2018023 \nDate: 31th July, 2024 \n \n \n \n \nThis is to certify that Mr. Mohd Ahkam was engaged as Associate with our organization as \nWeb Developer at Okhla, New Delhi from 05.07.2023 to 31.07.2024. \n \n \nHe was relieved from the engagement of the company on the above-mentioned date. \nHe had really shown good performance during his working period with our organization. \n \n \nWe wish good luck and success in all his future endeavors.

### Creating Data Chunks

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [14]:
chunks = split_documents(documents)
chunks

Split 1 documents into 1 chunks

Example chunk:
Content: This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG...
Metadata: {'source': '..\\data\\text_files\\python_intro.txt'}


[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]

### Embadding and VectorStoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

e:\Agentic AI\RAGS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class EmbeddingManager:
    """Handles document embedding using Sentence Transformers."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # all-MiniLM-L6-v2 is a popular model for sentence embeddings that balances performance and speed. and convert text to vectors for retrieval and generation tasks in RAG framework.
        """Initializing the embedding manager.
        Args:
            model_name: Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Loads the sentence transformer model."""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
        
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """Generates an embedding for the given text.
        Args:
            text: The input text to embed.
        returns:
            A numpy array of embedding with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for text: {len(texts)} texts...")  # Print the characters of the text
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"Embedding generated with shape: {embedding.shape}")
        return embedding
    
##Initializing the embedding manager and generating an embedding for a sample text
embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3725.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [11]:
class vector_store:
    """Manages document embeddings in the vector store using ChromaDB."""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initializing the vector store.
        Args:
            collection_name: Name of the ChromaDB collection to use.
            persist_directory: Directory where the ChromaDB will persist data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            # Create the persist chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Create the collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name,metadata={"description": "Pdf document embeddings for RAG"})
            print(f"Vector Store Initialized collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing Vector Store: {e}")
            raise
    
    def add_document(self, documents: List[Any], embeddings:np.ndarray):
        """Adds documents to the vector store with their embeddings.
        Args:
            documents: List of LangChain Document.
            embeddings: Corresponding embeddings for the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")
        
        # Prepare data for  ChromaDB
        ids = []
        metadatas = []
        texts = []
            
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Uniaue Id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # Generate a unique ID for each document
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)  # Convert metadata to a dictionary
            metadata['doc_index'] = i  # Add document index to metadata
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)  # Add embedding to metadata
            
            #Document Content
            documents_text.append(doc.page_content)
            
            # Embedding 
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for ChromaDB
            
        # Add to collection
        try:
            self.collection.add(ids=ids, embeddings=embeddings_list, metadatas=metadatas, documents=documents_text)
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to Vector Store: {e}")
            raise
        
vector_store=vector_store()
vector_store

Vector Store Initialized collection: pdf_documents
Existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]

In [16]:
texts = [chunk.page_content for chunk in chunks]
texts

['This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.']